In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score
from sklearn.impute import SimpleImputer
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Load data
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test_X.csv")

# Drop PatientID column
train_df = train_df.drop(columns=["PatientID"])
test_ids = test_df["PatientID"]
test_df = test_df.drop(columns=["PatientID"])

# Replace 0s with NaN for imputation
train_df['Cholesterol'] = train_df['Cholesterol'].replace(0, np.nan)
test_df['Cholesterol'] = test_df['Cholesterol'].replace(0, np.nan)
train_df['RestingBP'] = train_df['RestingBP'].replace(0, np.nan)
test_df['RestingBP'] = test_df['RestingBP'].replace(0, np.nan)

# Impute missing values
imputer = SimpleImputer(strategy='median')
train_df[['Cholesterol', 'RestingBP']] = imputer.fit_transform(train_df[['Cholesterol', 'RestingBP']])
test_df[['Cholesterol', 'RestingBP']] = imputer.transform(test_df[['Cholesterol', 'RestingBP']])

# Feature engineering
train_df['CholesterolPerAge'] = train_df['Cholesterol'] / train_df['Age']
test_df['CholesterolPerAge'] = test_df['Cholesterol'] / test_df['Age']

train_df['RestingBPPerAge'] = train_df['RestingBP'] / train_df['Age']
test_df['RestingBPPerAge'] = test_df['RestingBP'] / test_df['Age']

train_df['MaxHRPerAge'] = train_df['MaxHR'] / train_df['Age']
test_df['MaxHRPerAge'] = test_df['MaxHR'] / test_df['Age']

train_df['RestingBPPerCholesterol'] = train_df['RestingBP'] / train_df['Cholesterol']
test_df['RestingBPPerCholesterol'] = test_df['RestingBP'] / test_df['Cholesterol']

train_df['MaxHRPerRestingBP'] = train_df['MaxHR'] / train_df['RestingBP']
test_df['MaxHRPerRestingBP'] = test_df['MaxHR'] / test_df['RestingBP']

train_df['Oldpeak_significant'] = (train_df['Oldpeak'] > 0).astype(int)
test_df['Oldpeak_significant'] = (test_df['Oldpeak'] > 0).astype(int)

# Feature lists
categorical_features = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]
numerical_features = [
    "Age", "RestingBP", "Cholesterol", "FastingBS", "MaxHR", "Oldpeak",
    "CholesterolPerAge", "MaxHRPerAge", "RestingBPPerCholesterol"
]

# Preprocessing with RobustScaler
preprocessor = ColumnTransformer([
    ('num', RobustScaler(), numerical_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

# Base models (calibrated)
rf = CalibratedClassifierCV(RandomForestClassifier(
    class_weight='balanced', max_depth=10, min_samples_split=5,
    n_estimators=150, random_state=42), method='isotonic', cv=3)

svm = CalibratedClassifierCV(SVC(
    C=2, kernel='rbf', gamma='auto', degree=2,
    class_weight=None, probability=True), method='sigmoid', cv=3)

gb = CalibratedClassifierCV(GradientBoostingClassifier(
    n_estimators=150, max_depth=3, learning_rate=0.05,
    min_samples_split=20, min_samples_leaf=1,
    subsample=0.8, random_state=123), method='sigmoid', cv=3)

xgb = XGBClassifier(
    objective='binary:logistic', eval_metric='logloss',
    class_weight='balanced', max_depth=3, learning_rate=0.05,
    n_estimators=300, subsample=0.8, tree_method='exact', random_state=123)

lgbm = LGBMClassifier(
    learning_rate=0.05, max_depth=5, n_estimators=100,
    num_leaves=31, objective='binary',
    class_weight='balanced', random_state=42)

# Voting classifier with soft voting
voting_clf = VotingClassifier(
    estimators=[
        ('gb', gb),
        ('rf', rf),
        ('svm', svm),
        ('xgb', xgb),
        ('lgbm', lgbm)
    ],
    voting='soft'
)

# Final pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', voting_clf)
])

# Training data
X = train_df.drop(columns=["HeartDisease"])
y = train_df["HeartDisease"]

# Fit the model
pipeline.fit(X, y)

# Evaluate
train_preds = pipeline.predict(X)
train_f1 = f1_score(y, train_preds)
cv_scores = cross_val_score(pipeline, X, y, scoring='f1', cv=5)

print(f"Training F1 Score: {train_f1:.4f}")
print(f"Cross-Validation F1 Scores: {cv_scores}")
print(f"Mean CV F1 Score: {np.mean(cv_scores):.4f}")

# Predict test set
test_preds = pipeline.predict(test_df)

# Save to CSV
submission = pd.DataFrame({
    "PatientID": test_ids,
    "HeartDisease": test_preds
})
submission.to_csv("y_predict.csv", index=False)
print("Submission file saved as y_predict.csv")